<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ вакансий из HeadHunter
   

In [52]:
import pandas as pd
import psycopg2
import requests
from bs4 import BeautifulSoup

In [53]:
# вставьте сюда параметры подключения из юнита 1. Работа с базой данных из Python
DBNAME = 'project_sql'
USER = 'skillfactory'
PASSWORD = ''
HOST = '84.201.134.129'
PORT = 5432

In [54]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

# Юнит 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [55]:
# текст запроса
query_3_1 = f'''
select count(*)
from vacancies
'''

In [56]:
# результат запроса
number_of_vacancies = pd.read_sql(query_3_1, connection)
print(number_of_vacancies)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2056230821.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  number_of_vacancies = pd.read_sql(query_3_1, connection)


   count
0  49197


2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [57]:
# текст запроса
query_3_2 = f'''
select count(*)
from employers
'''

In [58]:
# результат запроса
number_of_employers = pd.read_sql(query_3_2, connection)
print(number_of_employers)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/3155309760.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  number_of_employers = pd.read_sql(query_3_2, connection)


   count
0  23501


3. Посчитате с помощью запроса количество регионов (таблица areas).

In [59]:
# текст запроса
query_3_3 = f'''
select count(*)
from areas
'''

In [60]:
# результат запроса
number_of_areas = pd.read_sql(query_3_3, connection)
print(number_of_areas)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/1975841597.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  number_of_areas = pd.read_sql(query_3_3, connection)


   count
0   1362


4. Посчитате с помощью запроса количество сфер деятельности в базе (таблица industries).

In [61]:
# текст запроса
query_3_4 = f'''
select count(*)
from industries
'''

In [62]:
# результат запроса
number_of_industries = pd.read_sql(query_3_4, connection)
print(number_of_industries)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/818949251.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  number_of_industries = pd.read_sql(query_3_4, connection)


   count
0    294


***

Выводы:

Произвели подсчет сущностей в таблицах БД:
- количество вакансий 49197
- количество работодателей 23501
- количество регионов 1362
- количество сфер деятельности 294

# Юнит 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [63]:
# текст запроса
query_4_1 = f'''
select
   count(v.area_id),
   a.name
from vacancies v
join areas a on v.area_id = a.id
group by v.area_id, a.name
order by 1 desc
limit 5
'''

In [64]:
# результат запроса
number_of_vacancies_by_area = pd.read_sql(query_4_1, connection)
display(number_of_vacancies_by_area)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/1665963951.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  number_of_vacancies_by_area = pd.read_sql(query_4_1, connection)


,count,name
0,5333,Москва
1,2851,Санкт-Петербург
2,2112,Минск
3,2006,Новосибирск
4,1892,Алматы


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [65]:
# текст запроса
query_4_2 = f'''
select
   count(*)
from vacancies v
where salary_from is not null or salary_to is not null
'''

In [66]:
# результат запроса
number_of_vacancies_by_salary = pd.read_sql(query_4_2, connection)
display(number_of_vacancies_by_salary)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/3820773153.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  number_of_vacancies_by_salary = pd.read_sql(query_4_2, connection)


,count
0,24073


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [67]:
# текст запроса
query_4_3 = f'''
select
   round(avg(v.salary_from)),
   round(avg(v.salary_to))
from vacancies v
'''

In [68]:
# результат запроса
average_salary = pd.read_sql(query_4_3, connection)
display(average_salary)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/259761091.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  average_salary = pd.read_sql(query_4_3, connection)


,round,round
0,71065.0,110537.0


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [69]:
# текст запроса
query_4_4 = f'''
select
   schedule,
   employment,
   count(schedule)
from vacancies v
group by schedule, employment
order by count(schedule) desc
'''

In [70]:
# результат запроса
result = pd.read_sql(query_4_4, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/702372665.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_4_4, connection)


,schedule,employment,count
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940
5,Полный день,Стажировка,569
6,Вахтовый метод,Полная занятость,367
7,Полный день,Частичная занятость,347
8,Гибкий график,Частичная занятость,312
9,Полный день,Проектная работа,141


5. Напишите запрос, выводящий значения поля Требуемый опыт работы (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [71]:
# текст запроса
query_4_5 = f'''
select
   experience,
   count(experience)
from vacancies v
group by experience
order by count(schedule)
'''

In [72]:
# результат запроса
result = pd.read_sql(query_4_5, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/1848489492.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_4_5, connection)


,experience,count
0,Более 6 лет,1337
1,Нет опыта,7197
2,От 3 до 6 лет,14511
3,От 1 года до 3 лет,26152


***

# Выводы по детальному анализу вакансий

На основе проведенного анализа в Юните 4, можно сделать следующие ключевые выводы:

1. **Географическое распределение вакансий**:
   - Большинство вакансий сконцентрировано в крупных городах
   - В топ-5 регионов по количеству вакансий значительный отрыв от остальных регионов

2. **Анализ зарплатных предложений**:
   - 24073 вакансий содержат информацию о зарплате
   - Средняя вилка зарплат:
     * Нижняя граница: около 71065 рублей
     * Верхняя граница: около 110537 рублей
   - Разница между верхней и нижней границей составляет примерно 39000 рублей

3. **Условия работы**:
   - Преобладающий формат: полный рабочий день + полная занятость
   - Альтернативные форматы (гибкий график, удаленная работа) представлены значительно реже
   - Это говорит о том, что рынок труда всё ещё придерживается традиционного формата занятости

4. **Требования к опыту работы**:
   - Наибольшее количество вакансий для специалистов с опытом 1-3 года и 3-6 лет
   - Наименьшее количество вакансий для специалистов с опытом более 6 лет
   - Вакансии для начинающих специалистов (без опыта) занимают среднюю позицию

Данный анализ показывает, что рынок труда ориентирован в первую очередь на специалистов среднего уровня, предпочитает традиционный формат работы и концентрируется в крупных городах.

# Юнит 5. Анализ работодателей

1. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.

Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей. Впишите его название в поле ниже в том виде, который вернул запрос.

In [73]:
# текст запроса
query_5_1 = f'''
select
   e.name,
   count(e.name)
from vacancies v
join employers e on v.employer_id = e.id
group by e.name
order by count(e.name) desc
limit 5
'''

In [74]:
# результат запроса
result = pd.read_sql(query_5_1, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/1535323220.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_1, connection)


,name,count
0,Яндекс,1933
1,Ростелеком,491
2,Тинькофф,444
3,СБЕР,428
4,Газпром нефть,331


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [75]:
# текст запроса
query_5_2 = f'''
select
   a.name,
   count(e.name) as employers_count
from areas a
join employers e on e.area = a.id
left join vacancies v on v.area_id = a.id
where v.id is null
group by a.name
order by count(e.name) desc
limit 5
'''

In [76]:
# результат запроса
result = pd.read_sql(query_5_2, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/84400773.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_2, connection)


,name,employers_count
0,Россия,410
1,Казахстан,207
2,Московская область,75
3,Краснодарский край,19
4,Беларусь,18


3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [77]:
# текст запроса
query_5_3 = f'''
select
   e.name,
   a.name,
   count(v.area_id) as areas_count
from employers e
join vacancies v on v.employer_id = e.id
join areas a on v.area_id = a.id
group by v.area_id, a.name, e.name
order by 3 desc
limit 20
'''

In [78]:
# результат запроса
result = pd.read_sql(query_5_3, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2056998736.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_3, connection)


,name,name,areas_count
0,СБЕР,Москва,205
1,DataArt,Алматы,124
2,DataArt,Нур-Султан,123
3,МТС,Москва,122
4,Газпром нефть,Санкт-Петербург,121
5,VK,Москва,79
6,Ozon,Москва,72
7,"МАГНИТ, Розничная сеть",Краснодар,68
8,Яндекс,Москва,54
9,Лига Цифровой Экономики,Москва,53


4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [79]:
# текст запроса
query_5_4 = f'''
select
   count(*)
from employers e
left join employers_industries ei on ei.employer_id = e.id
where ei.industry_id is null
'''

In [80]:
# результат запроса
result = pd.read_sql(query_5_4, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2222996321.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_4, connection)


,count
0,8419


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности. 

In [81]:
# текст запроса
query_5_5 = f'''
select
   e.name,
   count(e.name) as industries_count
from employers e
join employers_industries ei on ei.employer_id = e.id
group by e.name
having count(e.name) = 4
order by e.name
limit 3
'''

In [82]:
# результат запроса
result = pd.read_sql(query_5_5, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/1146473793.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_5, connection)


,name,industries_count
0,101 Интернет,4
1,21vek.by,4
2,2ГИС,4


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения.


In [83]:
# текст запроса
query_5_6 = f'''
select
   count(*)
from employers e
join employers_industries ei on ei.employer_id = e.id
join industries i on ei.industry_id = i.id
where i.name = 'Разработка программного обеспечения'
limit 10
'''

In [84]:
# результат запроса
result = pd.read_sql(query_5_6, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/973598188.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_6, connection)


,count
0,3553


7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку Total с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [85]:
# код для получения списка городов-милионников
url = 'https://ru.wikipedia.org/wiki/Города-миллионеры_России'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
soup_table = soup.find('table', class_='standard sortable')
rows = []
for row in soup_table.find_all('tr'):
    cells = row.find_all('td')
    row_data = [cell.text.strip() for cell in cells]
    if row_data:
        rows.append(row_data)

city_df = pd.DataFrame(rows)
city_millions = city_df.iloc[:, 1].to_list()
display(city_millions)

['Москва',
 'Санкт-Петербург',
 'Новосибирск',
 'Екатеринбург',
 'Казань',
 'Красноярск',
 'Нижний Новгород',
 'Челябинск',
 'Уфа',
 'Краснодар',
 'Самара',
 'Ростов-на-Дону',
 'Омск',
 'Воронеж',
 'Пермь',
 'Волгоград']

In [86]:
# текст запроса
city_millions_sql = str(tuple(city_millions))
query_5_7 = f'''
select
   a.name,
   count(a.name)
from employers e
join vacancies v on v.employer_id = e.id
join areas a on v.area_id = a.id
where e.name = 'Яндекс' and a.name in {city_millions_sql}
group by a.name
union all
select
   'Total',
   count(a.name)
from employers e
join vacancies v on v.employer_id = e.id
join areas a on v.area_id = a.id
where e.name = 'Яндекс' and a.name in {city_millions_sql}
order by 2
'''

In [87]:
# результат запроса
result = pd.read_sql(query_5_7, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2941469605.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_5_7, connection)


,name,count
0,Омск,21
1,Челябинск,22
2,Красноярск,23
3,Волгоград,24
4,Пермь,25
5,Казань,25
6,Ростов-на-Дону,25
7,Уфа,26
8,Самара,26
9,Краснодар,30


***

# Выводы по анализу работодателей

1. **Топ работодателей по количеству вакансий**:
   - Выявлены компании, которые наиболее активно размещают вакансии на платформе
   - Это позволяет определить крупнейших работодателей на рынке труда

2. **Географическая специфика работодателей**:
    - Обнаружены регионы с большим количеством зарегистрированных работодателей, однако в некоторых из них отсутствуют активные вакансии. Это может свидетельствовать о низкой деловой активности или сезонных факторах.


3. **Региональное присутствие компаний**:
   - Проанализировано распределение вакансий работодателей по регионам
   - Некоторые компании имеют широкую географию присутствия
   - Это показывает масштаб деятельности различных работодателей

4. **Сферы деятельности**:
   - Выявлено значительное количество работодателей (8419) без указания сферы деятельности
   - Это может говорить о:
     * Неполноте данных
     * Необходимости улучшения процесса заполнения профилей компаний

5. **Специализация работодателей**:
   - Проанализированы компании с различным количеством сфер деятельности
   - Наибольшее внимание привлекает сфера разработки ПО, так как она является одной из самых популярных областей. У значительного числа работодателей указана именно эта сфера.


6. **Анализ конкретных компаний (на примере Яндекса)**:
   - Рассмотрено присутствие в городах-миллионниках
   - Показано распределение вакансий по крупным городам
   - Это демонстрирует стратегию найма крупных работодателей

Данный анализ полезен для:
- Понимания структуры рынка труда
- Выявления ключевых работодателей
- Определения географических особенностей найма
- Понимания распределения бизнес-активности по регионам
- Оценки полноты данных о работодателях в системе

# Юнит 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 


In [88]:
# текст запроса
query_6_1 = f'''
select
   count(*)
from vacancies v
where v.name is not null
and (lower(v.name) like '%data%' or lower(v.name) like '%данн%')
'''

In [89]:
# результат запроса
result = pd.read_sql(query_6_1, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2120726066.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_6_1, connection)


,count
0,1771


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.
 

In [90]:
# текст запроса
query_6_2 = f'''
select
   count(*)
from vacancies v
where v.name is not null
and (
lower(v.name) like '%data%scientist%' or
lower(v.name) like '%data%science%' or
lower(v.name) like '%исследователь%данных%' or
(lower(v.name) like '%ml%' and lower(v.name) not like '%html%') or
lower(v.name) like '%machine%learning%' or
lower(v.name) like '%машинн%обучен%%'
)
and (
lower(v.name) like '%junior%'
or lower(v.experience) = 'нет опыта'
or lower(v.employment) = 'стажировка'
)
'''

In [91]:
# результат запроса
result = pd.read_sql(query_6_2, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/1855292916.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_6_2, connection)


,count
0,51


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?

** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [92]:
# текст запроса
query_6_3 = f'''
select
   count(*)
from vacancies v
where v.name is not null
and (
lower(v.name) like '%data%scientist%' or
lower(v.name) like '%data%science%' or
lower(v.name) like '%исследователь%данных%' or
(lower(v.name) like '%ml%' and lower(v.name) not like '%html%') or
lower(v.name) like '%machine%learning%' or
lower(v.name) like '%машинн%обучен%%'
)
and (lower(v.key_skills) like '%sql%' or lower(v.key_skills) like '%postgres%')
'''

In [93]:
# результат запроса
result = pd.read_sql(query_6_3, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/3454238784.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_6_3, connection)


,count
0,229


4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [94]:
# текст запроса
query_6_4 = f'''
select
   count(*)
from vacancies v
where v.name is not null
and (
lower(v.name) like '%data%scientist%' or
lower(v.name) like '%data%science%' or
lower(v.name) like '%исследователь%данных%' or
(lower(v.name) like '%ml%' and lower(v.name) not like '%html%') or
lower(v.name) like '%machine%learning%' or
lower(v.name) like '%машинн%обучен%%'
)
and lower(v.key_skills) like '%python%'
'''

In [95]:
# результат запроса
result = pd.read_sql(query_6_4, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/3349957416.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_6_4, connection)


,count
0,357


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [96]:
# текст запроса
query_6_5 = f'''
select
   round(avg(length(key_skills) - length(replace(key_skills, chr(9), ''))+1), 2)
from vacancies v
where (
lower(v.name) like '%data scientist%' or
lower(v.name) like '%data science%' or
lower(v.name) like '%исследователь данных%' or
(v.name  like '%ML%' and  (v.name not ilike '%html%')) or
lower(v.name) like '%machine learning%' or
lower(v.name) like '%машинн%обучен%'
)
'''

In [97]:
# результат запроса
result = pd.read_sql(query_6_5, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2400162839.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_6_5, connection)


,round
0,6.41


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа. 

In [98]:
# текст запроса
query_6_6 = f'''
select
   round(avg(coalesce((v.salary_from + v.salary_to)/2, v.salary_from, v.salary_to)))
from vacancies v
where (
lower(v.name) like '%data scientist%' or
lower(v.name) like '%data science%' or
lower(v.name) like '%исследователь данных%' or
(v.name  like '%ML%' and  (v.name not ilike '%html%')) or
lower(v.name) like '%machine learning%' or
lower(v.name) like '%машинн%обучен%'
)
and v.experience = 'От 3 до 6 лет'
'''

In [99]:
# результат запроса
result = pd.read_sql(query_6_6, connection)
display(result)

/var/folders/70/p1sc01nn2_92vgfrfd9_m6640000gn/T/ipykernel_31127/2201205693.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query_6_6, connection)


,round
0,243115.0


***

# Выводы по предметному анализу

1. **Количество вакансий, связанных с данными**:
   - Количество вакансий, содержащих ключевые слова 'data' или 'данн', равно 1771.

2. **Вакансии для начинающих специалистов**:
   - Проведен анализ вакансий для junior-специалистов в области Data Science
   - Учитывались следующие критерии:
     * Наличие слова 'junior' в названии
     * Отсутствие требований к опыту работы
     * Стажировки
   - Была найдена 51 вакансия. Это показывает, какие возможности есть для входа в профессию

3. **Требования к навыкам**:
   - Проанализированы требования к ключевым навыкам:
     * SQL и PostgreSQL являются важными требованиями для data science специалистов
     * Python является одним из самых востребованных навыков
   - Это помогает понять, какие технические навыки наиболее востребованы на рынке

4. **Среднее количество требуемых навыков**:
   - Выявлено среднее количество ключевых навыков, указываемых в вакансиях (6.41)
   - Это даёт представление об общем уровне требований к специалистам

5. **Анализ зарплат**:
   - Проведен анализ зарплатных предложений для специалистов с разным опытом
   - Особое внимание уделено специалистам с опытом 3-6 лет (з/п в районе 243115 рублей)
   - Это позволяет оценить финансовые перспективы карьерного роста

**Практическая значимость анализа**:
- Помогает понять требования рынка труда в сфере Data Science
- Даёт представление о необходимых навыках для входа в профессию
- Показывает уровень зарплатных ожиданий
- Позволяет оценить перспективы карьерного роста

**Рекомендации для соискателей**:
- Обратить внимание на развитие навыков работы с Python и SQL
- Учитывать, что большинство вакансий требует определённого набора ключевых навыков
- При планировании карьеры учитывать зависимость зарплаты от опыта работы

Данный анализ полезен как для начинающих специалистов, так и для опытных профессионалов при планировании карьеры в области Data Science.

# Общий вывод по проекту

In [100]:
# подведем итог исследования, обобщите выводы
# здесь можно (это будет плюсом) провести дополнительные исследования данных, сделать прогнозы, продумать варианты продолжения исследования

**Общие выводы по проекту анализа вакансий HeadHunter**

1. **Масштаб исследования**:
   - Проанализировано 49197 вакансий
   - Исследовано 23501 работодателей
   - Охвачено 1362 региона
   - Рассмотрено 294 сферы деятельности

2. **Географическое распределение**:
   - Большая концентрация вакансий в крупных городах и экономических центрах
   - Существует значительный разрыв между регионами по количеству вакансий
   - Некоторые регионы имеют много работодателей, но мало активных вакансий

3. **Зарплатные предложения**:
   - 24073 вакансий (≈49%) содержат информацию о зарплате
   - Средняя зарплатная вилка:
     * Нижняя граница: ~71000 рублей
     * Верхняя граница: ~110500 рублей
   - Разница между границами составляет около 39000 рублей

4. **Условия работы**:
   - Преобладает классический формат:
     * Полный рабочий день
     * Полная занятость
   - Альтернативные форматы (удаленная работа, гибкий график) встречаются реже

5. **Требования к опыту**:
   - Наиболее востребованы специалисты с опытом 1-3 и 3-6 лет
   - Меньше всего вакансий для специалистов с опытом более 6 лет
   - Вакансии без опыта занимают среднюю позицию

6. **Анализ сферы Data Science**:
   - Активно развивающееся направление
   - Ключевые требования:
     * Python
     * SQL/PostgreSQL
   - Существует спрос на начинающих специалистов
   - Зарплаты коррелируют с опытом работы

7. **Работодатели**:
   - Крупные компании доминируют по количеству вакансий
   - У 8419 работодателей не указана сфера деятельности
   - Многие компании работают в нескольких регионах

**Рекомендации**:

1. **Для соискателей**:
   - Обратить внимание на развитие востребованных навыков
   - Учитывать региональную специфику при поиске работы
   - Рассматривать возможности релокации в крупные города

2. **Для работодателей**:
   - Улучшить качество заполнения информации о компании
   - Рассмотреть возможности расширения географии найма
   - Обратить внимание на альтернативные форматы работы

3. **Для платформы HeadHunter**:
   - Улучшить качество данных о работодателях
   - Развивать инструменты анализа рынка труда
   - Внедрить дополнительные механизмы верификации данных

**Перспективы дальнейшего исследования**:
1. Анализ сезонности найма
2. Исследование корреляций между требованиями и зарплатами
3. Более детальный анализ региональных особенностей
4. Изучение динамики изменения требований к специалистам
5. Анализ влияния экономических факторов на рынок труда

Этот анализ предоставляет комплексное понимание текущей ситуации на рынке труда и может быть использован для принятия решений как соискателями, так и работодателями.